# 04 Query Cleaning C0 C1 C2

Kernel: **Python (NLP)**. This notebook uses only the frozen handoff files and saves derived outputs in the Phase 2 directories.

## Scope and approval gate

Compare original English questions (C0) with formatting-only normalization (C1) using the same pinned MiniLM document vectors, FAISS index, top-100 ranking, effective development QIDs, and metric code. The query review cell reads only QID and original question (plus game/task metadata for later grouping); it does not load answers, references, supplied retrieved docs, or test questions. C2 is not applied until the user reviews the exact correction inventory.

In [1]:
from pathlib import Path
import os, json, re, html, unicodedata, hashlib, time
from collections import defaultdict
import numpy as np
import pandas as pd
import faiss
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

HANDOFF_ROOT=Path.cwd().resolve()
while not (HANDOFF_ROOT/"02_FROZEN_DATA").is_dir():HANDOFF_ROOT=HANDOFF_ROOT.parent
PHASE_ROOT=HANDOFF_ROOT/"Phase2"; BUILD_ROOT=PHASE_ROOT/"DocumentBuilding"; CLEAN_ROOT=PHASE_ROOT/"CleaningExperiments"
OUT_DIR=CLEAN_ROOT/"outputs"; REPORT_DIR=CLEAN_ROOT/"reports"; LOG_DIR=CLEAN_ROOT/"logs"
HF_HOME=HANDOFF_ROOT/"NLP"/"hf_cache";os.environ["HF_HOME"]=str(HF_HOME)
for folder in (OUT_DIR,REPORT_DIR,LOG_DIR):folder.mkdir(parents=True,exist_ok=True)
MODEL_ID="sentence-transformers/all-MiniLM-L6-v2";MODEL_REVISION="1110a243fdf4706b3f48f1d95db1a4f5529b4d41";DEVICE="cpu"
QUERY_PATH=HANDOFF_ROOT/"02_FROZEN_DATA"/"Original_Queries_232_with_Usability_Fields.csv"
DEV_QID_PATH=BUILD_ROOT/"manifests"/"effective_development_qids.json"
CORPUS_PATH=HANDOFF_ROOT/"02_FROZEN_DATA"/"Corpus_4192_Gaming_Documents.csv"
QRELS_PATH=HANDOFF_ROOT/"02_FROZEN_DATA"/"Final_Relevance_Judgments_294_Pairs.csv"
INDEX_PATH=BUILD_ROOT/"indexes"/"chronoplay_minilm_faiss_index_v1.faiss"
MAPPING_PATH=BUILD_ROOT/"manifests"/"chronoplay_document_id_mapping_v1.csv"
CONFIG_PATH=BUILD_ROOT/"manifests"/"chronoplay_minilm_document_embedding_manifest_v1.json"
VALIDATION_PATH=BUILD_ROOT/"manifests"/"chronoplay_document_index_validation_v1.json"
assert json.loads(VALIDATION_PATH.read_text())["status"]=="PASS","Index validation did not pass; stop here."
print("Development-only; no test query or test assignment file is read.")

/Users/vaidhav/Desktop/MyProjects/NLP/ChronoPlay_Team_Handoff_v1/NLP/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1028 15:50:08.843000 9800 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Development-only; no test query or test assignment file is read.


In [2]:
# Select only original development question text and the metadata required for breakdowns. Answers, references and retrieved_docs are not read.
dev_info=json.loads(DEV_QID_PATH.read_text())
dev_qids=set(dev_info["qids"])
assert len(dev_qids)==188 and "181" not in dev_qids
queries=pd.read_csv(QUERY_PATH,usecols=["qid","question","game_name","task_type","usable"],dtype={"qid":"string"})
queries=queries.loc[queries["qid"].isin(dev_qids)].copy()
assert set(queries.qid)==dev_qids and len(queries)==188
assert queries["question"].notna().all()
queries["question_original"]=queries["question"].astype(str)
print("Effective development questions:",len(queries),"QID 181 excluded:","181" not in set(queries.qid))
# Display question-only review material to support the C2 inventory. No labels or answers enter this review.
display(queries[["qid","question_original"]].sort_values("qid",key=lambda s:s.astype(int)).reset_index(drop=True))

Effective development questions: 188 QID 181 excluded: True


,qid,question_original
0,0,Anyone playing Dune: Awakening 😍🔥 Been trying ...
1,2,I'm trying to figure out why I can't join the ...
2,3,I’m still getting the FG3 BattlEye Query Timeo...
3,5,How can I max out the size of my fief in Dune:...
4,6,Why is my Harvester collecting so little spice...
...,...,...
183,227,I'm mostly a solo player in PUBG Mobile—what k...
184,228,In my first match driving across the Erangel h...
185,229,We’re in squad mode and spot an airdrop landin...
186,230,"In PUBG Mobile's Erangel map, is the Military ..."


In [3]:
# C1 formatting-only normalization; record exactly which deterministic changes affected each question.
HTML_TAG_PATTERN=re.compile(r"</?[A-Za-z][^<>]{0,200}>")
def normalize_query_c1(value):
    original=str(value); changes=[]
    normalized=unicodedata.normalize("NFC",original)
    if normalized!=original: changes.append("Unicode NFC")
    decoded=html.unescape(normalized)
    if decoded!=normalized: changes.append("HTML entity decoding")
    without_markup=HTML_TAG_PATTERN.sub(" ",decoded)
    if without_markup!=decoded: changes.append("HTML tag removal")
    whitespace=" ".join(without_markup.split())
    if whitespace!=without_markup: changes.append("whitespace normalization")
    return whitespace,changes

queries["C0_query"]=queries["question_original"]
c1_results=[normalize_query_c1(q) for q in queries["question_original"]]
queries["C1_query"]=[x[0] for x in c1_results]
queries["C1_transformations"]=["; ".join(x[1]) if x[1] else "none" for x in c1_results]
print("C1 changed queries:",int((queries.C0_query!=queries.C1_query).sum()))
display(queries.loc[queries.C0_query!=queries.C1_query,["qid","C0_query","C1_query","C1_transformations"]].head(30))

C1 changed queries: 1


,qid,C0_query,C1_query,C1_transformations
194,194,Why is the PUBG MOBILE Version 3.6 update sche...,Why is the PUBG MOBILE Version 3.6 update sche...,whitespace normalization


In [4]:
# This list is populated only from defensible, manually reviewed question-only observations.
# Do not add entries based on answers, references, qrels, test queries, or guessed external aliases.
PROPOSED_C2_CORRECTIONS = []  # Full review of 188 original development questions found no defensible typo or alias correction.
INVENTORY_PATH=OUT_DIR/"c2_correction_inventory_proposed_v1.csv"
if INVENTORY_PATH.exists(): raise FileExistsError("Refusing to overwrite a prior C2 proposal inventory; version the filename.")
inventory_columns=["original_form","proposed_correction","affected_qids","transformation_type","justification","evidence_scope","deterministic"]
inventory=pd.DataFrame(PROPOSED_C2_CORRECTIONS,columns=inventory_columns)
inventory.to_csv(INVENTORY_PATH,index=False)
print("Proposed C2 inventory rows:",len(inventory),"saved:",INVENTORY_PATH)
if len(inventory): display(inventory)
else: print("No defensible correction has been entered. C2 correction application is blocked pending user direction.")
USER_REVIEWED_C2_INVENTORY=False
assert not USER_REVIEWED_C2_INVENTORY, "Set true only after the user has seen and approved the exact inventory."

Proposed C2 inventory rows: 0 saved: /Users/vaidhav/Desktop/MyProjects/NLP/ChronoPlay_Team_Handoff_v1/Phase2/CleaningExperiments/outputs/c2_correction_inventory_proposed_v1.csv
No defensible correction has been entered. C2 correction application is blocked pending user direction.


In [5]:
# Load the fixed validated model and exact FAISS index; C0/C1 share identical document vectors and ranking code.
snapshot_dir=Path(snapshot_download(repo_id=MODEL_ID,revision=MODEL_REVISION,cache_dir=str(HF_HOME),allow_patterns=["*.json","*.txt","*.safetensors"],local_files_only=True))
assert snapshot_dir.name==MODEL_REVISION
model=SentenceTransformer(str(snapshot_dir),device=DEVICE)
index=faiss.read_index(str(INDEX_PATH))
mapping=pd.read_csv(MAPPING_PATH,dtype={"document_id":"string"})
assert index.ntotal==4192 and mapping.document_id.nunique()==4192
assert json.loads(CONFIG_PATH.read_text())["model_revision"]==MODEL_REVISION
print("Model",MODEL_ID,MODEL_REVISION,"index",type(index).__name__,index.ntotal,"documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7474.54it/s]

Model sentence-transformers/all-MiniLM-L6-v2 1110a243fdf4706b3f48f1d95db1a4f5529b4d41 index IndexFlatIP 4192 documents


In [6]:
# Encode original and C1-cleaned questions using the same model and exact same reusable document index.
TOP_K=100
ranked_rows=[]
query_vectors_by_condition={}
latency_by_condition={}
for condition,text_column in (("C0","C0_query"),("C1","C1_query")):
    started=time.perf_counter()
    vectors=np.asarray(model.encode(queries[text_column].tolist(),batch_size=32,show_progress_bar=True,convert_to_numpy=True,normalize_embeddings=True,device=DEVICE),dtype=np.float32)
    scores,positions=index.search(vectors,TOP_K)
    elapsed=time.perf_counter()-started
    latency_by_condition[condition]={"total_seconds":elapsed,"mean_seconds":elapsed/len(queries)}
    query_vectors_by_condition[condition]=(scores,positions)
    for row_i,qid in enumerate(queries.qid):
        for rank,(score,pos) in enumerate(zip(scores[row_i],positions[row_i]),start=1):
            ranked_rows.append({"condition":condition,"query_id":qid,"document_id":str(mapping.iloc[int(pos)].document_id),"rank":rank,"score":float(score)})
print("Ranked rows:",len(ranked_rows),"per condition:",len(queries)*TOP_K)
assert len(ranked_rows)==2*len(queries)*TOP_K

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:  17%|█▋        | 1/6 [00:00<00:02,  1.96it/s]

Batches:  33%|███▎      | 2/6 [00:00<00:01,  3.51it/s]

Batches:  50%|█████     | 3/6 [00:00<00:00,  4.87it/s]

Batches:  83%|████████▎ | 5/6 [00:00<00:00,  7.41it/s]

...

Batches:  33%|███▎      | 2/6 [00:00<00:00,  6.54it/s]

Batches:  50%|█████     | 3/6 [00:00<00:00,  7.32it/s]

Batches:  83%|████████▎ | 5/6 [00:00<00:00,  9.81it/s]

Batches: 100%|██████████| 6/6 [00:00<00:00,  9.25it/s]

Ranked rows: 37600 per condition: 18800


In [7]:
# Load qrels only for scoring after query transformations and the proposed C2 inventory are fixed.
qrels=pd.read_csv(QRELS_PATH,usecols=["query_id","corpus_doc_id","relevance"],dtype={"query_id":"string","corpus_doc_id":"string"})
qrels=qrels.loc[qrels.query_id.isin(dev_qids)].copy()
assert len(qrels)==251 and set(qrels.query_id)==dev_qids
qrels["relevance"]=qrels["relevance"].astype(int)
assert set(qrels.relevance).issubset({1,2})
qrels_by_qid={qid:dict(zip(group.corpus_doc_id,group.relevance)) for qid,group in qrels.groupby("query_id")}
assert all(qid in qrels_by_qid for qid in dev_qids)
run_df=pd.DataFrame(ranked_rows)

def metric_for_query(ranked_ids,grades):
    top10=ranked_ids[:10]
    dcg=sum(((2**grades.get(doc,0))-1)/np.log2(rank+1) for rank,doc in enumerate(top10,1) if grades.get(doc,0)>0)
    ideal=sorted(grades.values(),reverse=True)[:10]
    idcg=sum(((2**grade)-1)/np.log2(rank+1) for rank,grade in enumerate(ideal,1))
    grade2={doc for doc,g in grades.items() if g==2}
    any_positive=set(grades)
    first2=next((i for i,doc in enumerate(ranked_ids[:10],1) if doc in grade2),None)
    return {
        "ndcg10":dcg/idcg if idcg else np.nan,
        "g2_mrr10":(1/first2 if first2 else (np.nan if not grade2 else 0.0)),
        "g2_recall5":(len(set(ranked_ids[:5])&grade2)/len(grade2) if grade2 else np.nan),
        "g2_recall10":(len(set(ranked_ids[:10])&grade2)/len(grade2) if grade2 else np.nan),
        "any_hit5":float(bool(set(ranked_ids[:5])&any_positive)),
        "any_hit10":float(bool(set(ranked_ids[:10])&any_positive)),
        "g1_retrieved10":sum(grades.get(doc)==1 for doc in ranked_ids[:10]),
        "g2_retrieved10":sum(grades.get(doc)==2 for doc in ranked_ids[:10]),
        "has_grade2_qrel":bool(grade2),
    }

per_query=[]
for condition,group in run_df.groupby("condition",sort=True):
    for qid,ranked_group in group.groupby("query_id",sort=False):
        ranked_ids=ranked_group.sort_values("rank").document_id.tolist()
        per_query.append({"condition":condition,"query_id":qid,**metric_for_query(ranked_ids,qrels_by_qid[qid])})
per_query_df=pd.DataFrame(per_query)
metric_columns=["ndcg10","g2_mrr10","g2_recall5","g2_recall10","any_hit5","any_hit10","g1_retrieved10","g2_retrieved10"]
overall=[]
for condition,group in per_query_df.groupby("condition"):
    row={"condition":condition,"n_queries":len(group),"n_queries_without_grade2_qrel":int((~group.has_grade2_qrel).sum())}
    for column in metric_columns:
        # Protocol/BM25-compatible macro: the Grade-2-only metrics assign zero where the query has no Grade 2 qrel.
        values=group[column].fillna(0) if column in ("g2_mrr10","g2_recall5","g2_recall10") else group[column]
        row[column]=float(values.mean())
        if column in ("g2_mrr10","g2_recall5","g2_recall10"):
            row[column+"_eligible_only"]=float(group.loc[group.has_grade2_qrel,column].mean())
    overall.append(row)
metrics_df=pd.DataFrame(overall)
display(metrics_df)

,condition,n_queries,n_queries_without_grade2_qrel,ndcg10,g2_mrr10,g2_mrr10_eligible_only,g2_recall5,g2_recall5_eligible_only,g2_recall10,g2_recall10_eligible_only,any_hit5,any_hit10,g1_retrieved10,g2_retrieved10
0,C0,188,2,0.640672,0.611753,0.618331,0.693308,0.700763,0.730701,0.738558,0.739362,0.771277,0.106383,0.835106
1,C1,188,2,0.640672,0.611753,0.618331,0.693308,0.700763,0.730701,0.738558,0.739362,0.771277,0.106383,0.835106


In [8]:
# Group breakdowns and paired bootstrap differences; query IDs are paired across C0 and C1.
query_meta=queries.set_index("qid")[["game_name","task_type"]]
per_query_df["game_name"]=per_query_df.query_id.map(query_meta.game_name)
per_query_df["task_type"]=per_query_df.query_id.map(query_meta.task_type)
group_rows=[]
for field in ("game_name","task_type"):
    for value,subset in per_query_df.groupby(field):
        for condition,g in subset.groupby("condition"):
            item={"group_type":field,"group":value,"condition":condition,"n_queries":len(g)}
            for col in metric_columns:
                vals=g[col].fillna(0) if col in ("g2_mrr10","g2_recall5","g2_recall10") else g[col]
                item[col]=float(vals.mean())
            group_rows.append(item)
group_metrics_df=pd.DataFrame(group_rows)
display(group_metrics_df)

# Bootstrap paired mean differences C1 minus C0 for all-query metrics, with a fixed reproducible seed.
wide=per_query_df.pivot(index="query_id",columns="condition",values=metric_columns)
rng=np.random.default_rng(42);N_BOOTSTRAP=10000;paired_rows=[]
for metric in metric_columns:
    if metric in ("g2_mrr10","g2_recall5","g2_recall10"):
        c0=wide[(metric,"C0")].fillna(0).to_numpy();c1=wide[(metric,"C1")].fillna(0).to_numpy()
    else:
        c0=wide[(metric,"C0")].to_numpy();c1=wide[(metric,"C1")].to_numpy()
    diffs=c1-c0
    sample_indices=rng.integers(0,len(diffs),size=(N_BOOTSTRAP,len(diffs)))
    boot=diffs[sample_indices].mean(axis=1)
    paired_rows.append({"comparison":"C1-C0","metric":metric,"n_queries":len(diffs),"mean_difference":float(diffs.mean()),"ci95_low":float(np.quantile(boot,.025)),"ci95_high":float(np.quantile(boot,.975)),"bootstrap_seed":42,"bootstrap_replicates":N_BOOTSTRAP})
paired_df=pd.DataFrame(paired_rows)
display(paired_df)

,group_type,group,condition,n_queries,ndcg10,g2_mrr10,g2_recall5,g2_recall10,any_hit5,any_hit10,g1_retrieved10,g2_retrieved10
0,game_name,dune,C0,41,0.466656,0.460801,0.532729,0.570044,0.634146,0.658537,0.195122,0.902439
1,game_name,dune,C1,41,0.466656,0.460801,0.532729,0.570044,0.634146,0.658537,0.195122,0.902439
2,game_name,dyinglight2,C0,73,0.514667,0.485921,0.561644,0.595890,0.602740,0.630137,0.027397,0.643836
3,game_name,dyinglight2,C1,73,0.514667,0.485921,0.561644,0.595890,0.602740,0.630137,0.027397,0.643836
4,game_name,pubgm,C0,74,0.861389,0.819520,0.912162,0.952703,0.932432,0.972973,0.135135,0.986486
5,game_name,pubgm,C1,74,0.861389,0.819520,0.912162,0.952703,0.932432,0.972973,0.135135,0.986486
6,task_type,Comparative QA,C0,19,0.680494,0.684211,0.728520,0.756410,0.842105,0.842105,0.421053,1.263158
7,task_type,Comparative QA,C1,19,0.680494,0.684211,0.728520,0.756410,0.842105,0.842105,0.421053,1.263158
8,task_type,Extraction-based QA,C0,135,0.632538,0.609536,0.700000,0.725926,0.733333,0.755556,0.059259,0.792593
9,task_type,Extraction-based QA,C1,135,0.632538,0.609536,0.700000,0.725926,0.733333,0.755556,0.059259,0.792593


,comparison,metric,n_queries,mean_difference,ci95_low,ci95_high,bootstrap_seed,bootstrap_replicates
0,C1-C0,ndcg10,188,0.0,0.0,0.0,42,10000
1,C1-C0,g2_mrr10,188,0.0,0.0,0.0,42,10000
2,C1-C0,g2_recall5,188,0.0,0.0,0.0,42,10000
3,C1-C0,g2_recall10,188,0.0,0.0,0.0,42,10000
4,C1-C0,any_hit5,188,0.0,0.0,0.0,42,10000
5,C1-C0,any_hit10,188,0.0,0.0,0.0,42,10000
6,C1-C0,g1_retrieved10,188,0.0,0.0,0.0,42,10000
7,C1-C0,g2_retrieved10,188,0.0,0.0,0.0,42,10000


In [9]:
# Save C0/C1 outputs. C2 stays blocked until the exact proposal inventory is reviewed by the user.
CLEANED_PATH=OUT_DIR/"cleaned_queries_C0_C1_v1.csv"
RUN_PATH=OUT_DIR/"dense_ranked_run_C0_C1_v1.csv"
PER_QUERY_PATH=OUT_DIR/"per_query_metrics_C0_C1_v1.csv"
METRICS_PATH=OUT_DIR/"metrics_comparison_C0_C1_v1.csv"
GROUP_PATH=OUT_DIR/"metrics_by_game_task_C0_C1_v1.csv"
PAIRED_PATH=OUT_DIR/"paired_differences_C1_vs_C0_v1.csv"
EXPERIMENT_MANIFEST=OUT_DIR/"experiment_manifest_C0_C1_v1.json"
for path in (CLEANED_PATH,RUN_PATH,PER_QUERY_PATH,METRICS_PATH,GROUP_PATH,PAIRED_PATH,EXPERIMENT_MANIFEST):
    if path.exists():raise FileExistsError(f"Refusing to overwrite {path}; increment version.")
cleaned_rows=[]
for _,r in queries.iterrows():
    cleaned_rows.extend([
        {"query_id":r.qid,"condition":"C0","original_query":r.question_original,"transformed_query":r.C0_query,"applied_transformations":"none"},
        {"query_id":r.qid,"condition":"C1","original_query":r.question_original,"transformed_query":r.C1_query,"applied_transformations":r.C1_transformations},
    ])
pd.DataFrame(cleaned_rows).to_csv(CLEANED_PATH,index=False)
run_df.to_csv(RUN_PATH,index=False)
per_query_df.to_csv(PER_QUERY_PATH,index=False)
metrics_df.to_csv(METRICS_PATH,index=False)
group_metrics_df.to_csv(GROUP_PATH,index=False)
paired_df.to_csv(PAIRED_PATH,index=False)
manifest={
 "status":"C0_C1_COMPLETE_C2_REVIEW_PENDING","dataset":"ChronoPlay-QA Claude Verified v1.1",
 "model_id":MODEL_ID,"model_revision":MODEL_REVISION,"device":DEVICE,"index_file":str(INDEX_PATH.relative_to(HANDOFF_ROOT)),
 "development_queries":len(queries),"development_qrels":len(qrels),"excluded_qids":["181"],"test_evaluated":False,
 "conditions":{"C0":"original English question","C1":"Unicode NFC, HTML entity/tag handling only when present, whitespace collapse","C2":"not applied; correction inventory requires user review"},
 "top_k":TOP_K,"metric_definitions":"protocol graded nDCG gains 3/1; Grade2-only MRR/Recall; any-positive hit; macro by query; no-grade2 query metrics reported as undefined plus BM25-compatible macro-as-zero fields",
 "bootstrap":{"seed":42,"replicates":N_BOOTSTRAP,"paired":"C1 minus C0"},"latency_seconds":latency_by_condition,
 "c2_inventory_file":str(INVENTORY_PATH.relative_to(HANDOFF_ROOT)),"c2_user_reviewed":False,
 "outputs":[str(p.relative_to(HANDOFF_ROOT)) for p in (CLEANED_PATH,RUN_PATH,PER_QUERY_PATH,METRICS_PATH,GROUP_PATH,PAIRED_PATH)],
}
EXPERIMENT_MANIFEST.write_text(json.dumps(manifest,indent=2)+"\n",encoding="utf-8")
print("Saved C0/C1 runs and metrics. C2 remains unapplied pending user review.")
for path in (CLEANED_PATH,RUN_PATH,PER_QUERY_PATH,METRICS_PATH,GROUP_PATH,PAIRED_PATH,EXPERIMENT_MANIFEST,INVENTORY_PATH):print(path.relative_to(HANDOFF_ROOT),path.stat().st_size,"bytes")

Saved C0/C1 runs and metrics. C2 remains unapplied pending user review.
Phase2/CleaningExperiments/outputs/cleaned_queries_C0_C1_v1.csv 119028 bytes
Phase2/CleaningExperiments/outputs/dense_ranked_run_C0_C1_v1.csv 2476928 bytes
Phase2/CleaningExperiments/outputs/per_query_metrics_C0_C1_v1.csv 27577 bytes
Phase2/CleaningExperiments/outputs/metrics_comparison_C0_C1_v1.csv 648 bytes
Phase2/CleaningExperiments/outputs/metrics_by_game_task_C0_C1_v1.csv 2271 bytes
Phase2/CleaningExperiments/outputs/paired_differences_C1_vs_C0_v1.csv 435 bytes
Phase2/CleaningExperiments/outputs/experiment_manifest_C0_C1_v1.json 1813 bytes
Phase2/CleaningExperiments/outputs/c2_correction_inventory_proposed_v1.csv 111 bytes
